<div dir="rtl" style="text-align: right;">

**RAG** از دو پایپ‌لاین ساخته شده: <bdi>Indexing</bdi> (آفلاین، سنگین، یک‌بار) و <bdi>Query</bdi> (آنلاین، سریع، هر سؤال).

- <bdi>Indexing</bdi>: <bdi>Load → Clean → Chunk → Embed → Store</bdi>
- <bdi>Query</bdi>: <bdi>Embed Query → Retrieve → Build Context → Prompt → Generate → Sources</bdi>
- اسناد و سؤال باید با یک مدل <bdi>embedding</bdi> تبدیل بشن.
- <bdi>Retriever</bdi> و <bdi>Generator</bdi> مستقل هستن و جداگانه باید تست بشن.
- بیشتر خطاها از <bdi>Retrieval</bdi> میان، نه از <bdi>LLM</bdi>.
- <bdi>Metadata</bdi> باید از ابتدا تا انتها همراه <bdi>chunk</bdi> باشه (<bdi>citation</bdi>، فیلتر، دسترسی).
- در پروژه‌ی <bdi>HR</bdi>: <bdi>Indexing</bdi> = آیین‌نامه‌ها و <bdi>FAQ</bdi> ها؛ <bdi>Query</bdi> = سؤال کارمندها همراه فیلتر دسترسی.

</div>

In [1]:
from dataclasses import dataclass, field

@dataclass
class Chunk:
    id: str
    text: str
    metadata: dict = field(default_factory=dict)   # source, page, dept, ...
    embedding: list[float] | None = None

# ---------- Indexing Pipeline (آفلاین / batch) ----------
class IndexingPipeline:
    def __init__(self, loader, cleaner, chunker, embedder, store):
        self.loader, self.cleaner = loader, cleaner
        self.chunker, self.embedder, self.store = chunker, embedder, store

    def run(self, paths: list[str]) -> int:
        total = 0
        for path in paths:
            doc = self.loader.load(path)               # Phase 2
            doc.text = self.cleaner.clean(doc.text)    # Phase 2
            chunks = self.chunker.split(doc)           # Phase 3
            vectors = self.embedder.embed_documents([c.text for c in chunks])  # Phase 4
            for c, v in zip(chunks, vectors):
                c.embedding = v
            self.store.upsert(chunks)                  # Phase 5
            total += len(chunks)
        return total

# ---------- Query Pipeline (آنلاین / realtime) ----------
class QueryPipeline:
    def __init__(self, embedder, store, llm, top_k=5):
        self.embedder, self.store = embedder, store
        self.llm, self.top_k = llm, top_k

    def answer(self, question: str, user_ctx: dict) -> dict:
        q_vec = self.embedder.embed_query(question)                     # Phase 4
        hits = self.store.search(q_vec, k=self.top_k,
                                 filters={"dept": user_ctx["allowed_depts"]})  # Phase 5
        if not hits:
            return {"answer": "اطلاعاتی در این مورد پیدا نشد.", "sources": []}
        context = "\n\n".join(f"[{i+1}] {h.text}" for i, h in enumerate(hits))   # Phase 6
        prompt = build_prompt(context, question)                        # Phase 6
        answer = self.llm.generate(prompt)
        return {"answer": answer,
                "sources": [h.metadata for h in hits]}